# Module 3: Reinforcement Learning — MDP Concepts

In Modules 1 and 2 you built the *perception* half of the pipeline: webcam → landmarks → state vector.  
Now we focus on the *decision-making* half: given a state vector, how does an agent choose its next action?

The answer is **Reinforcement Learning (RL)**, and the mathematical language of RL is the **Markov Decision Process (MDP)**.

---

## What is a Markov Decision Process?

An MDP is a formal model of sequential decision-making. It defines five things:

| Symbol | Name | Meaning |
|--------|------|---------|
| **S** | State space | The set of all possible observations the agent can receive |
| **A** | Action space | The set of all actions the agent can take |
| **R(s, a)** | Reward function | A scalar signal the environment returns after each step |
| **T(s′ \| s, a)** | Transition function | The probability of landing in state s′ when taking action a in state s |
| **γ** | Discount factor | How much the agent values future rewards relative to immediate ones (0 < γ ≤ 1) |

The **Markov property** says: the next state depends *only* on the current state and action — not on the full history. This is what makes the math tractable.

### The policy

A **policy** π(a | s) is the agent's decision rule: a mapping from states to (a distribution over) actions. The goal of RL training is to find the policy that maximises the expected cumulative discounted reward:

```
G_t = R_{t+1} + γ·R_{t+2} + γ²·R_{t+3} + …
```

Throughout this module we use **CartPole-v1** as our concrete MDP example.

In [ ]:
# Concepts based on: https://huggingface.co/learn/deep-rl-course (MIT, low-maintenance reference)

import sys
from pathlib import Path

# Locate repo root — walk up from the notebook's directory until we find utils/gym_utils.py.
# This works whether the notebook is run via nbconvert (CWD = repo root or notebook dir)
# or interactively in VS Code / Jupyter Lab (CWD varies).
_search = Path.cwd()
for _ in range(6):
    if (_search / "utils" / "gym_utils.py").exists():
        REPO_ROOT = _search
        break
    _search = _search.parent
else:
    raise FileNotFoundError(
        "Could not find repo root (utils/gym_utils.py not found in any parent directory). "
        "Run this notebook from inside the physical-ai-workshop repository."
    )

sys.path.insert(0, str(REPO_ROOT))

from utils.gym_utils import make_env

# Create CartPole-v1 — use render_mode="rgb_array" to avoid display errors
# in headless nbconvert execution (no screen required)
env = make_env("CartPole-v1", render_mode="rgb_array")

print("=== MDP: State and Action Spaces ===")
print()
print("Observation space (S):", env.observation_space)
print("  → Shape:", env.observation_space.shape)
print("  → Low: ", env.observation_space.low)
print("  → High:", env.observation_space.high)
print()
print("Action space (A):", env.action_space)
print("  → Number of discrete actions:", env.action_space.n)

## The State Vector

CartPole's **observation space** is a continuous 4-element vector — this is the MDP state **s**:

| Index | Variable | Description |
|-------|----------|-------------|
| 0 | `cart_position` | Horizontal position of the cart (metres, range ≈ ±4.8) |
| 1 | `cart_velocity` | Horizontal velocity of the cart |
| 2 | `pole_angle` | Angle of the pole from vertical (radians, range ≈ ±0.418) |
| 3 | `pole_angular_velocity` | Rate of change of the pole angle |

The episode ends (the pole is considered *fallen*) when `|pole_angle| > 0.2095 rad` (~12°) or `|cart_position| > 2.4 m`. This boundary condition defines the **terminal states** of the MDP.

In [ ]:
# Concepts based on: https://huggingface.co/learn/deep-rl-course (MIT, low-maintenance reference)

import numpy as np

# Reset the environment to get the initial state s_0
obs, info = env.reset(seed=42)

print("Initial observation (state s_0):")
print(f"  cart_position          = {obs[0]:+.4f} m")
print(f"  cart_velocity          = {obs[1]:+.4f} m/s")
print(f"  pole_angle             = {obs[2]:+.4f} rad  ({np.degrees(obs[2]):+.2f}°)")
print(f"  pole_angular_velocity  = {obs[3]:+.4f} rad/s")
print()
print("Type:", type(obs))
print("Shape:", obs.shape)
print("Dtype:", obs.dtype)

## Actions and Rewards

CartPole's **action space** is discrete with 2 choices:

| Action | Meaning |
|--------|---------|
| `0` | Push cart to the **left** |
| `1` | Push cart to the **right** |

The **reward function** R(s, a) is simple:
- **+1** for every timestep the pole remains upright
- **0** on the terminal step (when the episode ends)

This sparse, constant reward is deceptively tricky — the agent receives no gradient signal *during* the episode telling it which actions were better. RL algorithms like PPO must figure this out through trial and error across many episodes.

The maximum possible return for a single episode is **500** (CartPole-v1's step limit).

In [ ]:
# Concepts based on: https://huggingface.co/learn/deep-rl-course (MIT, low-maintenance reference)

# Take a few random actions to observe the (s, a, r, s') tuple — the basic
# unit of experience in RL

print("Stepping through the MDP with random actions:")
print(f"{'Step':>4}  {'Action':>6}  {'Reward':>6}  {'Done':>5}  state")
print("-" * 70)

obs, info = env.reset(seed=0)

for step in range(8):
    action = env.action_space.sample()   # random policy
    next_obs, reward, terminated, truncated, info = env.step(action)

    done = terminated or truncated
    state_str = f"[{next_obs[0]:+.3f}, {next_obs[1]:+.3f}, {next_obs[2]:+.3f}, {next_obs[3]:+.3f}]"
    action_str = "LEFT " if action == 0 else "RIGHT"
    print(f"{step + 1:>4}  {action_str:>6}  {reward:>6.1f}  {str(done):>5}  {state_str}")

    obs = next_obs
    if done:
        print("  ↳ Episode ended (pole fell or cart left bounds)")
        break

print()
print("Each row is one (s, a, r, s') transition — the elementary experience tuple.")

## The Transition Function

The **transition function** T(s′ | s, a) defines how the world evolves. For CartPole it is **deterministic** — given the same state and action, you always get the same next state:

```
T(s′ | s, a) = 1   for exactly one s′
             = 0   for all other s′
```

This is governed by classical mechanics (Newton's second law applied to the pole-on-cart system). In contrast, real-world robotic environments are **stochastic**: motor noise, sensor noise, and unmodelled dynamics mean T(s′ | s, a) is a proper probability distribution over next states.

### Why does the episode reset?

When `terminated=True`, the environment has reached an absorbing state — the pole has fallen beyond recovery. Gymnasium's convention is that `env.reset()` is then called to start a fresh episode. From the MDP perspective, the agent transitions to a special **terminal state** from which the only possible transition is to the initial state distribution (via reset).

### Sim-to-real gap

Policies trained in simulation (like the PPO agent you will train in the next script) must be robust to the stochasticity and modelling errors not present in the simulator. This **sim-to-real gap** is one of the central challenges of Physical AI — and a key reason Domain Randomisation and Foundation Model reasoning layers are so valuable.

In [ ]:
# Concepts based on: https://huggingface.co/learn/deep-rl-course (MIT, low-maintenance reference)

# Demonstrate deterministic transitions: same (s, a) always produces the same s'.
# We reset twice with the same seed, apply the same action, and compare next states.
obs_a, _ = env.reset(seed=99)
next_a, reward_a, *_ = env.step(1)  # push right — trial 1

env.reset(seed=99)  # restore identical starting state
next_b, reward_b, *_ = env.step(1)  # same action — trial 2

print("Deterministic transition check:")
print(f"  Starting state (seed=99): {obs_a}")
print(f"  Action applied:           1 (RIGHT)")
print(f"  Next state (trial 1):     {next_a}")
print(f"  Next state (trial 2):     {next_b}")
print(f"  States identical: {np.allclose(next_a, next_b)}")
print()
print("CartPole is deterministic — same (s, a) always produces the same s'.")
print("Real physical systems add noise here, making RL harder (and more interesting).")

env.close()
print()
print("Environment closed. Ready for Module 3 — training a PPO agent!")